# 第 08 章：工程防御：安全断点与状态注入 实战

根据讲义，本章实验主要验证：
- 引入 `checkpointer` 持久化，并配置 `interrupt_before` 断点。
- 手动干预流程，体验 HITL。
- 验证 `InjectedState` 是如何在背后默默起作用的。

In [ ]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

# 如果 api.deepseek.com 无法访问，可以尝试修改 base_url 或使用代理
llm = init_chat_model(
    model="deepseek-chat",
    model_provider="openai",
    base_url="https://api.deepseek.com",
    streaming=True
)
print(f"模型加载成功：{llm.__class__.__name__}")

## 1. 核心实战：构建带断点的审批流 Agent

In [ ]:
from typing import TypedDict, Annotated, Sequence
import operator
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, ToolMessage
from langchain.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver

class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]

@tool
def delete_database(table_name: str) -> str:
    """删除核心数据库表。高危操作！！！"""
    return f"成功删除表 {table_name}! 如果这是真的，你已经被解雇了。"

tools = [delete_database]
llm_with_tools = llm.bind_tools(tools)

def call_model(state: AgentState):
    messages = state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

# 建立图结构
workflow = StateGraph(AgentState)
workflow.add_node("agent", call_model)
workflow.add_node("tools", ToolNode(tools))

workflow.add_edge(START, "agent")
workflow.add_conditional_edges("agent", tools_condition)
workflow.add_edge("tools", "agent")

# [核心配置] 引入内存，设置断点
memory_saver = MemorySaver()
app = workflow.compile(
    checkpointer=memory_saver,
    interrupt_before=["tools"]  # 在准备执行工具前，挂起图的执行
)
print("代理图编译完毕，带挂起点。")

### 执行并遇见断点 (Trigger the breakpoint)

In [ ]:
config = {"configurable": {"thread_id": "demo_thread_01"}}

# 我们向他下达一个危险的命令
user_msg = HumanMessage(content="帮我删除所有的 user 表。")

print("🟢 开始首轮执行...")
for event in app.stream({"messages": [user_msg]}, config, stream_mode="values"):
    print("当前步骤:", [m.type for m in event["messages"]])

print("\n🛑 运行结束。检查当前状态:")
current_state = app.get_state(config)
print("接下来的节点 (next):", current_state.next)

你会发现，图并没有跑完，而是停在了 `tools` 节点之前。
### 手动放行 (Resume)
如果管理员确认无误，想要批准执行，我们只需要用空的内容唤醒它即可。

In [ ]:
# 管理员拒绝的话，可以用 app.update_state 注入一条拒绝信息。
# 演示直接放行（由于上一个状态已经是挂起，送入 None 即可恢复）
print("🟢 管理员审批通过，放行！")
for event in app.stream(None, config, stream_mode="values"):
    last_msg = event["messages"][-1]
    print("最新节点产出:", last_msg.type, getattr(last_msg, "content", ""))

## 2. InjectedState 依赖倒置隔离

In [ ]:
from langgraph.prebuilt import InjectedState

# 在真正的工业场景中，很多需要被隔离的信息（如用户 ID, Auth Token）是放在 State 里的
class ContextState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]
    user_role: str
    
@tool
def secret_action(action_name: str, u_role: Annotated[str, InjectedState("user_role")]) -> str:
    """执行机密操作。"""
    if u_role == "admin":
        return f"管理员权限确认， {action_name} 成功执行。"
    else:
        return f"拦截：你只是一个 {u_role}，无权执行 {action_name}。"

# 对于 LLM 来说，它只能考究到 action_name 这个参数，u_role 是隐藏的。
print("工具对外暴露的 Schema: ", secret_action.args_schema.schema())
